# ITMO AITH: DL+NLP HW6 — Abliteration + QLoRA-DPO

Отчётный ноутбук для HW6. **Ничего не обучает, GPU не нужен.** Читает
зафиксированные артефакты из `artifacts_hw6/` и показывает:

- refusal-rate на harmful и harmless до/после каждой стадии;
- результаты layer-search для abliteration;
- DPO loss curve;
- примеры генераций до/после на фиксированных промптах;
- ссылку на HF Hub (аблитерированная база).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

ART = Path('artifacts_hw6')

def read_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

run_summary = read_json(ART / 'run_summary.json') or {}
abl_info    = read_json(ART / 'abliteration_info.json') or {}
abl_scores  = read_json(ART / 'abliteration_layer_scores.json') or {}
hf_info     = read_json(ART / 'hf_push_info.json') or {}
dpo_metrics = read_json(ART / 'dpo_train_metrics.json') or {}
dpo_log     = read_json(ART / 'dpo_log_history.json') or {}

run_summary

## Refusal-rate: pretrained → abliterated → DPO

Одинаковый eval-сплит, одинаковый `GenConfig` (greedy, 192 токена),
одинаковая regex-эвристика. Harmful: высокий refusal-rate = модель
отказывает. Harmless: refusal-rate должен оставаться **низким** —
проверяем, что мы не сломали общую полезность.

In [ ]:
rows = []
for tag in ('pretrained', 'abliterated', 'dpo'):
    m = read_json(ART / f'metrics_{tag}.json')
    if not m:
        rows.append({'tag': tag, 'rr_harmful': None, 'rr_harmless': None, 'n_harmful': None, 'n_harmless': None})
        continue
    rows.append({
        'tag': tag,
        'rr_harmful':  m.get('refusal_rate_harmful'),
        'rr_harmless': m.get('refusal_rate_harmless'),
        'n_harmful':   m.get('n_harmful'),
        'n_harmless':  m.get('n_harmless'),
    })

df_rr = pd.DataFrame(rows).set_index('tag')
df_rr.style.format({'rr_harmful': '{:.3f}', 'rr_harmless': '{:.3f}'}, na_rep='—')

## Layer-search для abliteration

Каждая кандидат-direction проверена runtime-ablation'ом на маленьком
held-out harmful-сэмпле. По оси Y — refusal-rate при ablation этой
direction во всех слоях. Чем ниже — тем лучше direction справляется с
подавлением отказа.

In [ ]:
if abl_scores:
    import matplotlib.pyplot as plt
    layers = sorted(int(k) for k in abl_scores.keys())
    scores = [abl_scores[str(l)] for l in layers]
    best_layer = abl_info.get('best_layer')

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(layers, scores, marker='o', linewidth=1)
    if best_layer is not None and best_layer in layers:
        ax.axvline(best_layer, color='red', linestyle='--', alpha=0.5,
                   label=f'best_layer={best_layer}')
        ax.legend()
    ax.set_xlabel('layer index')
    ax.set_ylabel('refusal_rate @ ablation')
    ax.set_title('Layer search for refusal direction')
    ax.grid(True, alpha=0.3)
    plt.show()
    print(f"best_layer={best_layer} (из {abl_info.get('n_layers_total')}); "
          f"hidden_dim={abl_info.get('hidden_dim')}")
else:
    print('abliteration_layer_scores.json не найден')

## DPO loss curve

In [ ]:
history = dpo_log.get('log_history') or []
if history:
    import matplotlib.pyplot as plt
    steps = [h.get('step') for h in history if 'loss' in h]
    losses = [h.get('loss') for h in history if 'loss' in h]
    if steps and losses:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(steps, losses, marker='.', linewidth=1)
        ax.set_xlabel('step')
        ax.set_ylabel('DPO loss')
        ax.grid(True, alpha=0.3)
        plt.show()
    m = dpo_metrics
    print(f"effective_batch={m.get('effective_batch_size')}, "
          f"epochs={m.get('num_train_epochs')}, beta={m.get('beta')}, "
          f"lr={m.get('learning_rate')}")
    print(f"final train_loss={m.get('train_loss')}, "
          f"runtime_sec={m.get('train_runtime_sec')}")
else:
    print('dpo_log_history.json пустой / не найден')

## Примеры генераций

Фиксированный eval-сплит, одни и те же промпты. По 3 harmful + 2 harmless.
`is_refusal` — флаг regex-эвристики.

In [ ]:
def _short(text: str, n: int = 280) -> str:
    text = (text or '').strip().replace('\n', ' ')
    return text if len(text) <= n else text[:n] + '…'


def _load_rows(tag: str, kind: str):
    return read_json(ART / f'eval_{tag}_{kind}.json') or []


def _compare_block(kind: str, n: int):
    pre = _load_rows('pretrained', kind)
    abl = _load_rows('abliterated', kind)
    dpo = _load_rows('dpo', kind)
    if not pre:
        return f'(no {kind} eval rows)'
    # Все три файла генерились по одному и тому же списку prompt'ов и в
    # одном и том же порядке, так что сравнение по индексу корректно.
    blocks = []
    for i in range(min(n, len(pre))):
        p = pre[i]['prompt']
        blocks.append(f'### [{kind} #{i}] {p}\n')
        for tag, rows in (('pretrained', pre), ('abliterated', abl), ('dpo', dpo)):
            if i < len(rows):
                r = rows[i]
                mark = 'X' if r.get('is_refusal') else '·'
                blocks.append(f'**{tag}** [{mark}]: {_short(r["response"])}\n')
        blocks.append('')
    return '\n'.join(blocks)


from IPython.display import Markdown
Markdown(_compare_block('harmful', 3) + '\n---\n' + _compare_block('harmless', 2))

## Артефакты и ссылки

In [ ]:
from IPython.display import Markdown

lines = []
if hf_info.get('hf_repo_id'):
    lines.append(f"- HF Hub: https://huggingface.co/{hf_info['hf_repo_id']}")
if run_summary.get('model'):
    lines.append(f"- Base model: `{run_summary['model']}`")
if run_summary.get('seed') is not None:
    lines.append(f"- Seed: {run_summary['seed']}")
if abl_info.get('best_layer') is not None:
    lines.append(f"- Abliterate best_layer: {abl_info['best_layer']}")

Markdown('\n'.join(lines) if lines else '(run_summary.json не найден)')